# 01 — Diagnostic des familles de features Gradient Boosting

Ce notebook charge uniquement les artefacts OOF de la référence et de l'étude `GB_FEATURE_FAMILY_STUDY`. Il n'entraîne aucun modèle, n'accède à aucune cible de test et ne calcule aucune métrique sur le lockbox.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
def trouver_racine_depot(depart: Path) -> Path:
    for candidat in (depart, *depart.parents):
        if (candidat / 'pyproject.toml').is_file():
            return candidat
    raise FileNotFoundError('Impossible de localiser la racine du dépôt.')


racine_depot = trouver_racine_depot(Path.cwd().resolve())
dossier_reference = racine_depot / 'artifacts' / 'experiments' / 'GB_RET20_REFERENCE'
dossier_etude = racine_depot / 'artifacts' / 'experiments' / 'GB_FEATURE_FAMILY_STUDY'
comparaison = pd.read_csv(dossier_etude / 'comparison.csv')
inventaire = pd.read_csv(dossier_etude / 'feature_inventory.csv')
with (dossier_etude / 'study_summary.json').open(encoding='utf-8') as fichier:
    resume_etude = json.load(fichier)
with (dossier_reference / 'summary.json').open(encoding='utf-8') as fichier:
    resume_reference = json.load(fichier)

## Identité de l'étude et gate locale

In [ ]:
pd.Series({
    'référence immuable': resume_etude['reference']['experiment_id'],
    'périmètre': resume_etude['evaluation_scope'],
    'candidat retenu': resume_etude['selected_submission_candidate'],
    'gate seconde soumission': resume_etude['second_submission_gate_passed'],
    'métriques lockbox calculées': resume_etude['lockbox_metrics_computed'],
})

## Comparaison OOF des familles de features

In [ ]:
colonnes = [
    'experiment_id', 'accuracy', 'accuracy_gain',
    'bootstrap_ci95_lower', 'bootstrap_ci95_upper',
    'non_negative_fold_count', 'roc_auc', 'log_loss',
    'run_1_fit_time_seconds', 'admissible',
]
comparaison[colonnes].sort_values('accuracy', ascending=False)

In [ ]:
ordonnee = comparaison.sort_values('accuracy_gain')
erreurs = [
    ordonnee['accuracy_gain'] - ordonnee['bootstrap_ci95_lower'],
    ordonnee['bootstrap_ci95_upper'] - ordonnee['accuracy_gain'],
]
ax = ordonnee.plot.barh(
    x='experiment_id', y='accuracy_gain', xerr=erreurs,
    figsize=(10, 6), legend=False,
)
ax.axvline(0.0, color='black', linestyle='--')
ax.set_xlabel('Gain d’accuracy OOF par rapport à RET20')
ax.set_ylabel('Expérience')
ax.set_title('Gain apparié et intervalle bootstrap à 95 %')
plt.tight_layout()

## Inventaire et valeurs manquantes

In [ ]:
inventaire[['name', 'family', 'dtype', 'n_unique', 'missing_rate', 'in_train', 'in_test']].sort_values(['family', 'name'])

## Stabilité par fold du candidat retenu

In [ ]:
candidat = resume_etude['selected_submission_candidate']
metriques_candidat = pd.read_csv(dossier_etude / candidat / 'fold_metrics.csv')
metriques_reference = pd.read_csv(dossier_reference / 'fold_metrics.csv')
folds = metriques_candidat[['fold_id', 'accuracy', 'roc_auc', 'log_loss']].merge(
    metriques_reference[['fold_id', 'accuracy']], on='fold_id',
    suffixes=('_candidat', '_reference'), validate='one_to_one',
)
folds['gain_accuracy'] = folds['accuracy_candidat'] - folds['accuracy_reference']
folds

## Conclusion

Le turnover complète utilement les rendements. L'ajout fold-safe de `GROUP` au meilleur ensemble numérique apporte un gain OOF robuste. Les volumes, les indicateurs de valeurs manquantes et `ALLOCATION` ne sont pas retenus. Cette conclusion concerne exclusivement les OOF de développement.